# **Label-free CBM и LaBo: горлышко из концептов без разметки**

Практика к модулю [«Concept-based методы»](https://ai-interpretability.school), урок «Concept Bottleneck Models (CBM)», шаги «Концепты без ручной разметки: Label-free CBM» и «Концепты без ручной разметки: LaBo».

В тетради `PRACTICE_cbm_cem` концепты XOR и dot data были размечены заранее, и CBM училась их предсказывать. У настоящих изображений такой разметки обычно нет. Label-free CBM ([Oikarinen et al., ICLR 2023](https://arxiv.org/abs/2304.06129)) и LaBo ([Yang et al., CVPR 2023](https://arxiv.org/abs/2211.11158)) заменяют ее парой готовых моделей: список концептов пишет языковая модель, а насколько концепт виден на снимке, решает CLIP. Здесь мы соберем оба метода своим кодом на `torch` поверх CLIP ViT-B/32 и CIFAR-10:

1. данные: подвыборка CIFAR-10, признаки сети-основы ResNet-18 и эмбеддинги CLIP;
2. список концептов GPT-3 из репозитория Label-free CBM и фильтры концептов;
3. слой концептов Label-free CBM: проекция признаков ResNet-18, обученная повторять сходства CLIP;
4. разреженный последний слой с elastic net и его чувствительность к силе разреженности;
5. LaBo: отбор концептов для каждого класса и матрица весов «классы × концепты»;
6. сравнение точности и числа ненулевых весов последнего слоя с CBM из `PRACTICE_cbm_cem`, CLIP без обучения и линейными классификаторами над признаками;
7. где методы ломаются.

Концепты взяты готовым файлом, GPT-3 не вызывается. Обе модели (ResNet-18 около 45 МБ и CLIP около 600 МБ) и CIFAR-10 (около 170 МБ) скачиваются при первом запуске без ключей. С GPU (Runtime → Change runtime type → T4 GPU) кодирование 7000 снимков занимает около минуты, на процессоре Colab — около 10 минут; остальная тетрадь идет за пару минут.

In [ ]:
!pip install -q transformers==5.18.0

In [ ]:
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm без ipywidgets; в Colab ipywidgets есть
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import transformers
from torchvision import transforms
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
print(device, torch.__version__, transformers.__version__)

## 1. Данные

CIFAR-10 — 10 классов, снимки 32×32. Чтобы тетрадь шла и на процессоре, возьмем подвыборку с фиксированным сидом: 400 снимков каждого класса для обучения, еще 100 каждого класса из обучающей части для валидации и 200 каждого класса из тестовой части, всего 4000, 1000 и 2000 снимков.

Каждый снимок пройдет через две сети:

- **сеть-основа** — ResNet-18, обученная на ImageNet, без последнего слоя: 512 признаков. Ее признаки Label-free CBM проецирует в концепты; CLIP в ней не участвует;
- **CLIP ViT-B/32** — нормированный эмбеддинг изображения, тоже 512 чисел. С ним считаются сходства «изображение — текст концепта».

Label-free CBM в статье для CIFAR-10 тоже берет сеть-основу и CLIP разными сетями. Если взять сетью-основой сам CLIP, слой концептов станет тривиальным: сходство с текстом — линейная функция того же эмбеддинга.

In [ ]:
train_set = torchvision.datasets.CIFAR10('data', train=True, download=True)
test_set = torchvision.datasets.CIFAR10('data', train=False, download=True)
CLASSES = train_set.classes
rng = np.random.default_rng(SEED)


def per_class(targets, n):
    # по n случайных индексов каждого класса
    y = np.array(targets)
    return [rng.permutation(np.where(y == k)[0])[:n] for k in range(10)]


blocks = per_class(train_set.targets, 500)
idx = {'train': np.concatenate([b[:400] for b in blocks]),
       'val': np.concatenate([b[400:] for b in blocks]),
       'test': np.concatenate(per_class(test_set.targets, 200))}
source = {'train': train_set, 'val': train_set, 'test': test_set}
y = {s: torch.tensor(np.array(source[s].targets)[idx[s]], device=device) for s in idx}
print(CLASSES)
print({s: len(idx[s]) for s in idx})

In [ ]:
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
resnet.fc = torch.nn.Identity()          # 512 признаков предпоследнего слоя
resnet_prep = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


@torch.no_grad()
def encode_images(split, batch=250):
    ds, ids = source[split], idx[split]
    clip_emb, res_feat = [], []
    for s in range(0, len(ids), batch):
        images = [ds[i][0] for i in ids[s:s + batch]]
        pixels = processor(images=images, return_tensors='pt')['pixel_values'].to(device)
        clip_emb.append(F.normalize(clip.get_image_features(pixel_values=pixels).pooler_output, dim=-1))
        res_feat.append(resnet(torch.stack([resnet_prep(im) for im in images]).to(device)))
    return torch.cat(clip_emb), torch.cat(res_feat)


@torch.no_grad()
def encode_text(texts):
    tokens = tokenizer(texts, padding=True, return_tensors='pt').to(device)
    return F.normalize(clip.get_text_features(**tokens).pooler_output, dim=-1)


X, B = {}, {}                            # X — эмбеддинги CLIP, B — признаки ResNet-18
for s in idx:
    X[s], B[s] = encode_images(s)
mu, sd = B['train'].mean(0), B['train'].std(0)
B = {s: (B[s] - mu) / sd for s in B}     # стандартизация по обучающей части
print({s: tuple(X[s].shape) for s in X}, {s: tuple(B[s].shape) for s in B})

Две точки сравнения, которые не объясняют свой прогноз концептами: CLIP без обучения (zero-shot: класс, чей текст `a photo of a {класс}` ближе всего к снимку) и линейные классификаторы над признаками ResNet-18 и над эмбеддингом CLIP. Классификатор обучаем той же функцией `fit_linear`, что дальше пойдет для последнего слоя концептов: полный батч, кросс-энтропия, градиентный спуск.

In [ ]:
def fit_linear(Z, labels, steps=2000, lr=0.1):
    torch.manual_seed(SEED)
    W = torch.zeros(10, Z.shape[1], device=device, requires_grad=True)
    b = torch.zeros(10, device=device, requires_grad=True)
    opt = torch.optim.SGD([W, b], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        F.cross_entropy(Z @ W.T + b, labels).backward()
        opt.step()
    return W.detach(), b.detach()


def accuracy(Z, W, b, labels):
    return ((Z @ W.T + b).argmax(1) == labels).float().mean().item()


T_prompt = encode_text([f'a photo of a {c}' for c in CLASSES])
results = {}
results['CLIP zero-shot'] = (((X['test'] @ T_prompt.T).argmax(1) == y['test']).float().mean().item(), 0)
W, b = fit_linear(B['train'], y['train'])
results['линейный классификатор над ResNet-18'] = (accuracy(B['test'], W, b, y['test']), int((W != 0).sum()))
W, b = fit_linear(10 * X['train'], y['train'])   # ×10: эмбеддинг CLIP нормирован, без множителя спуск идет медленно
results['линейный классификатор над CLIP'] = (accuracy(10 * X['test'], W, b, y['test']), int((W != 0).sum()))
for k, (a, n) in results.items():
    print(f'{k:45s} точность {a:.4f}   ненулевых весов {n}')

## 2. Список концептов и фильтры

Для CIFAR-10 авторы Label-free CBM задали GPT-3 три вопроса о каждом классе и выложили ответы в репозиторий: `important` — по каким признакам класс узнают, `around` — что чаще всего видно рядом, `superclass` — к каким более общим категориям класс относится. Возьмем эти три файла как есть.

In [ ]:
URL = 'https://raw.githubusercontent.com/Trustworthy-ML-Lab/Label-free-CBM/main/data/concept_sets/gpt3_init/gpt3_cifar10_{}.json'
gpt = {q: json.loads(urllib.request.urlopen(URL.format(q), timeout=30).read()) for q in ('important', 'around', 'superclass')}
for q in gpt:
    print(q, '|', 'airplane:', gpt[q]['airplane'])

**Задание 1. Соберите словарь `pools`: для каждого класса — отсортированный список его концептов из трех файлов без повторов, строки в нижнем регистре и без пробелов по краям. Соберите `concepts` — отсортированный список всех концептов всех классов без повторов. Сколько строк в трех файлах вместе и сколько в `concepts` разных концептов?**

In [ ]:
# Ваш код здесь

Label-free CBM чистит список четырьмя фильтрами, три из них смотрят только на текст:

1. **длина**: концепт длиннее 30 символов убирают, его трудно прочесть в объяснении;
2. **сходство с названием класса**: концепт, который почти повторяет название класса (`a cat` у класса `cat`), убирают, иначе горлышко пропускает сам ответ;
3. **сходство с другим концептом**: из пары почти одинаковых концептов (`a bird feeder` и `a birdfeeder`) оставляют один.

Сходство здесь — косинус между эмбеддингами текстов CLIP. Авторы считают его еще и второй языковой моделью (all-mpnet-base-v2) и берут пороги 0.85 и 0.9; у нас одна CLIP, а ее эмбеддинги коротких текстов вида `a ...` близки друг к другу почти всегда, поэтому пороги выше: 0.90 для названий классов и 0.95 для пар концептов.

**Задание 2. Напишите функцию `text_filters(concepts, class_cut=0.90, pair_cut=0.95)`. Она убирает концепты длиннее 30 символов; затем концепты, у которых наибольший косинус с эмбеддингом названия какого-либо класса (`encode_text(CLASSES)`) больше `class_cut`; затем идет по оставшимся в порядке «сначала короткие, при равной длине по алфавиту» и оставляет концепт, только если его косинус с каждым уже оставленным меньше `pair_cut`. Функция печатает число концептов после каждого фильтра и возвращает список оставленных. Сколько концептов остается после трех фильтров?**

In [ ]:
# Ваш код здесь

Четвертый фильтр смотрит на снимки: **присутствие**. Для каждого концепта берут пять обучающих снимков, на которых косинус «эмбеддинг снимка — эмбеддинг текста» в CLIP наибольший, и усредняют эти пять чисел. Если среднее не выше 0.25 (порог авторов), CLIP почти не находит концепт ни на одном снимке, и концепт убирают.

In [ ]:
def presence_filter(names, cut=0.25, verbose=True):
    top5 = (X['train'] @ encode_text(names).T).topk(5, dim=0).values.mean(0)
    if verbose:
        print('убраны:', [n for n, t in zip(names, top5) if t <= cut])
    return [n for n, t in zip(names, top5) if t > cut]


after_presence = presence_filter(after_text)
print('осталось:', len(after_presence))

## 3. Слой концептов Label-free CBM

Пусть $f(x)\in\mathbb{R}^{512}$ — признаки ResNet-18, $P_{ij}$ — косинус CLIP между снимком $i$ и концептом $j$. Слой концептов — линейная проекция $W_c\in\mathbb{R}^{K\times 512}$ (здесь $K$ означает число концептов, в уроке оно обозначено $M$, а $K$ там означает число классов), и ее обучают так, чтобы столбец активаций концепта $j$ по всем снимкам, $q_j = (W_c f(x_i))_{i}$, повторял столбец $P_{:,j}$ с точностью до сдвига и масштаба. Мерой служит косинус кубов:

$$\mathrm{sim}(q_j, P_{:,j}) = \cos\big(\overline{q_j^{\,3}},\ \overline{P_{:,j}^{\,3}}\big),$$

где куб берется поэлементно, а черта значит «вычесть среднее по снимкам». Куб усиливает самые большие значения: важнее всего, чтобы активация концепта была высокой на тех снимках, где CLIP видит его лучше всего. Функция потерь — $-\frac{1}{K}\sum_j \mathrm{sim}(q_j, P_{:,j})$ на обучающих снимках.

**Задание 3. Напишите функцию `cos_cubed(a, b)`: для матриц $a, b$ размера «снимки × концепты» она возвращает вектор длины $K$ — косинус между столбцами $a^3$ и $b^3$, из которых вычтено среднее по снимкам.**

In [ ]:
# Ваш код здесь

Проекцию обучаем Adam полным батчем до 1000 шагов и оставляем веса с лучшим средним $\mathrm{sim}$ на валидации: на 4000 снимках проекция 512 → $K$ быстро переобучается. Последний фильтр — **интерпретируемость**: концепт, у которого $\mathrm{sim}$ на валидации не выше 0.45 (порог авторов), слой концептов повторяет плохо, и его убирают. Затем активации оставшихся концептов стандартизуют по обучающей части: это и есть горлышко $c(x)$.

In [ ]:
def concept_layer(names, steps=1000, lr=1e-3, cut=0.45, verbose=True):
    T = encode_text(names)
    P_train, P_val = X['train'] @ T.T, X['val'] @ T.T
    torch.manual_seed(SEED)
    Wc = torch.nn.Linear(512, len(names), bias=False).to(device)
    opt = torch.optim.Adam(Wc.parameters(), lr=lr)
    best, best_W, best_step = -1.0, None, 0
    for step in range(1, steps + 1):
        opt.zero_grad()
        (-cos_cubed(Wc(B['train']), P_train).mean()).backward()
        opt.step()
        if step % 50 == 0:
            with torch.no_grad():
                v = cos_cubed(Wc(B['val']), P_val).mean().item()
            if v > best:
                best, best_W, best_step = v, Wc.weight.detach().clone(), step
    with torch.no_grad():
        sim_val = cos_cubed(B['val'] @ best_W.T, P_val)
    good = sim_val > cut
    if verbose:
        print(f'лучший шаг {best_step}, средний sim на валидации {best:.3f}')
        print('убраны фильтром интерпретируемости:', [n for n, g in zip(names, good) if not g])
    W_keep = best_W[good]
    C = {s: B[s] @ W_keep.T for s in B}
    m, s_ = C['train'].mean(0), C['train'].std(0)
    return [n for n, g in zip(names, good) if g], {s: (C[s] - m) / s_ for s in C}, sim_val[good]


lf_names, C, lf_sim = concept_layer(after_presence)
print('концептов в горлышке:', len(lf_names))

## 4. Разреженный последний слой

Классы предсказывает линейный слой $\hat y = \arg\max_k (W c(x) + b)_k$, и его учат с elastic net:

$$\min_{W,b}\ \mathrm{CE}(Wc+b,\ y) + \lambda\Big(\alpha\,\|W\|_1 + \frac{1-\alpha}{2}\,\|W\|_2^2\Big),\qquad \alpha=0.99.$$

Слагаемое $\|W\|_1$ недифференцируемо в нуле, поэтому после обычного шага градиента по гладкой части делают **проксимальный шаг**: каждый вес сдвигают к нулю на $\eta\lambda\alpha$ ($\eta$ — шаг обучения), а вес, который перешел бы через ноль, обнуляют: $w \leftarrow \mathrm{sign}(w)\max(|w|-\eta\lambda\alpha,\ 0)$. Так веса становятся ровно нулевыми, а не просто маленькими. Авторы решают ту же задачу пакетом GLM-SAGA; полный батч на 4000 снимках проще сделать руками.

**Задание 4. Напишите функцию `fit_sparse(Z, labels, lam, alpha=0.99, steps=2000, lr=0.1)`: веса и сдвиг с нулей, полный батч, на каждом шаге градиент кросс-энтропии плюс $\lambda\frac{1-\alpha}{2}\|W\|_2^2$, шаг спуска для $W$ и $b$ и проксимальный шаг для $W$. Возвращает `W, b`. Обучите ее на `C['train']` для каждого $\lambda$ из списка `LAMS` и заполните таблицу: точность на тесте и число ненулевых весов.**

In [ ]:
# Ваш код здесь

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep.iloc[:, 2], sweep.iloc[:, 1], 'o-', color='#2a6f97')
for _, r in sweep.iterrows():
    ax.annotate(f"λ={r['lambda']}", (r.iloc[2], r.iloc[1]), textcoords='offset points', xytext=(5, -12), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('nonzero weights of the last layer (log scale)')
ax.set_ylabel('test accuracy')
ax.set_title('Label-free CBM: accuracy against sparsity')
ax.grid(alpha=0.3)
plt.show()

Дальше берем $\lambda=0.003$: у каждого класса остается около двух десятков ненулевых весов, а точность на тесте в прогоне решения ниже, чем у слоя без разреженности, примерно на 2 пункта. Посмотрим, на какие концепты опирается каждый класс: три концепта с наибольшими положительными весами.

In [ ]:
LAM = 0.003
W_lf, b_lf = fit_sparse(C['train'], y['train'], LAM)
results['Label-free CBM, λ=0'] = (accuracy(C['test'], *fit_sparse(C['train'], y['train'], 0.0), y['test']), len(lf_names) * 10)
results[f'Label-free CBM, λ={LAM}'] = (accuracy(C['test'], W_lf, b_lf, y['test']), int((W_lf != 0).sum()))


def top_concepts(W, names, k=3):
    return pd.DataFrame({c: [f'{names[j]} ({W[i, j]:.2f})' for j in W[i].argsort(descending=True)[:k].tolist()]
                         for i, c in enumerate(CLASSES)}).T


top_concepts(W_lf, lf_names)

Объяснение одного прогноза — вклады концептов в логит предсказанного класса: $W_{\hat y j}\,c_j(x)$. У разреженного слоя ненулевых слагаемых около двух десятков, и восемь крупнейших по модулю уже показывают, из чего сложился логит.

In [ ]:
i = 7                                    # номер снимка в тестовой подвыборке
logits = C['test'][i] @ W_lf.T + b_lf
pred = logits.argmax().item()
contrib = W_lf[pred] * C['test'][i]
order = contrib.abs().argsort(descending=True)[:8].tolist()[::-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={'width_ratios': [1, 2.4]})
axes[0].imshow(test_set[idx['test'][i]][0])
axes[0].set_title(f'true: {CLASSES[y["test"][i]]}, predicted: {CLASSES[pred]}', fontsize=9)
axes[0].axis('off')
axes[1].barh([lf_names[j] for j in order], contrib[order].cpu(), color=['#2a6f97' if v > 0 else '#c0392b' for v in contrib[order]])
axes[1].set_xlabel(f'contribution W[{CLASSES[pred]}, j] · c_j(x) to the logit')
axes[1].set_title('eight largest concept contributions', fontsize=9)
fig.tight_layout()
plt.show()

Значение концепта горлышка равно сходству, которое считает CLIP. Проверить это можно только глазами: покажем шесть тестовых снимков с наибольшей активацией двух концептов, на которые сильнее всего опираются классы `horse` и `ship`.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for row, cls in enumerate(('horse', 'ship')):
    j = W_lf[CLASSES.index(cls)].argmax().item()
    for col, t in enumerate(C['test'][:, j].argsort(descending=True)[:6].tolist()):
        axes[row, col].imshow(test_set[idx['test'][t]][0])
        axes[row, col].set_title(CLASSES[y['test'][t]], fontsize=8)
        axes[row, col].axis('off')
    axes[row, 0].text(-4, 16, lf_names[j], ha='right', va='center', fontsize=9)
fig.suptitle('six test images with the largest activation of the concept (title: true class)', fontsize=10)
fig.tight_layout()
plt.show()

## 5. LaBo

LaBo устроен иначе. Горлышко не обучается: значение концепта $j$ на снимке — косинус CLIP $g_j(x) = \langle e_{\text{img}}(x), e_{\text{text}}(j)\rangle$. Обучается только матрица «классы × концепты», и в начале обучения каждый концепт связан только со своим классом.

**Отбор концептов.** Для каждого класса из его концептов выбирают $k$ так, чтобы они одновременно отличали класс от других и не повторяли друг друга. Авторы максимизируют жадно субмодулярную функцию из двух слагаемых:

- **различимость** $D(j)$: средний по снимкам класса косинус с концептом $j$ сравнивают с теми же средними у других классов; у нас это доля класса в softmax по десяти классам с температурой 0.01;
- **покрытие**: для каждого концепта пула берут его наибольший косинус с уже выбранными концептами и усредняют по пулу. Это слагаемое растет медленнее, если новый концепт похож на уже выбранные.

На каждом шаге добавляют концепт, который больше всего увеличивает сумму $\sum_{j\in S} D(j) + \text{покрытие}(S)$. Пул класса — его концепты из трех файлов, из которых убраны почти повторяющие название класса (тот же фильтр 0.90). Для сравнения тот же размер $k$ набирают случайно.

In [ ]:
class_means = torch.stack([X['train'][y['train'] == k].mean(0) for k in range(10)])
labo_pool = {}
for c in CLASSES:
    T = encode_text(pools[c])
    keep = (T @ T_class.T).max(1).values <= 0.90
    labo_pool[c] = ([n for n, kk in zip(pools[c], keep) if kk], T[keep])
print({c: len(labo_pool[c][0]) for c in CLASSES})


def select_concepts(k, random=False):
    names, embs, owner = [], [], []
    g = torch.Generator().manual_seed(SEED)
    for ci, c in enumerate(CLASSES):
        pool, T = labo_pool[c]
        if random:
            chosen = torch.randperm(len(pool), generator=g)[:k].tolist()
        else:
            D = torch.softmax((class_means @ T.T) / 0.01, dim=0)[ci]
            sim = T @ T.T
            chosen = []
            for _ in range(min(k, len(pool))):
                gain = [D[chosen + [j]].sum() + sim[:, chosen + [j]].max(1).values.mean() if j not in chosen else -1e9
                        for j in range(len(pool))]
                chosen.append(int(torch.tensor(gain).argmax()))
        names += [pool[j] for j in chosen]
        embs.append(T[chosen])
        owner += [ci] * len(chosen)
    return names, torch.cat(embs), torch.tensor(owner, device=device)


labo_names, labo_T, labo_owner = select_concepts(5)
pd.DataFrame({c: labo_names[5 * i:5 * i + 5] for i, c in enumerate(CLASSES)}).T

**Матрица весов.** Пусть $g(x)\in\mathbb{R}^{K}$ — косинусы снимка с отобранными концептами, стандартизованные по обучающей части. Веса — матрица $A\in\mathbb{R}^{10\times K}$, у которой каждая строка — распределение по концептам: $A = \mathrm{softmax}(V)$ по строкам, где обучается $V$. Логит класса — $\tau\,\langle A_k, g(x)\rangle$ с множителем $\tau=10$. Языковое априорное знание входит через начало обучения: $V_{kj} = 5$, если концепт $j$ отобран для класса $k$, и 0 иначе.

**Задание 5. Напишите функцию `fit_labo(T, owner, steps=500, lr=0.01)`: по эмбеддингам текстов отобранных концептов `T` и номерам их классов `owner` она строит $g$ для обучающей и тестовой частей, учит $V$ Adam полным батчем на кросс-энтропии и возвращает точность на тесте и матрицу $A$. Посчитайте точность LaBo с $k=1, 3, 5, 10$ концептами на класс при субмодулярном и случайном отборе.**

In [ ]:
# Ваш код здесь

Матрица $A$ после обучения: какая доля веса каждого класса осталась на его собственных концептах. В начале обучения $V_{kj}=5$ дает каждому своему концепту вес $e^5/(5e^5+45)\approx 0.19$, и на своих пяти концептах лежит около 0.94 веса строки.

In [ ]:
acc_labo, A_labo = fit_labo(labo_T, labo_owner)
results['LaBo, 5 концептов на класс'] = (acc_labo, int((A_labo != 0).sum()))
own = (A_labo * F.one_hot(labo_owner, 10).T).sum(1)
print('доля веса на своих концептах по классам:', {c: round(v, 3) for c, v in zip(CLASSES, own.tolist())})

fig, ax = plt.subplots(figsize=(11, 3.4))
im = ax.imshow(A_labo.cpu(), aspect='auto', cmap='Blues')
ax.set_yticks(range(10), CLASSES, fontsize=8)
ax.set_xticks(range(0, 50, 5), [CLASSES[i] for i in range(10)], fontsize=7, rotation=45, ha='left')
ax.set_xlabel('selected concepts, grouped by the class they were selected for')
ax.set_title('LaBo: weight matrix A after training (row = class)')
fig.colorbar(im, ax=ax, fraction=0.02)
fig.tight_layout()
plt.show()

## 6. Сравнение с CBM из `PRACTICE_cbm_cem`

CBM из тетради `PRACTICE_cbm_cem` — две части, `concept_encoder` и `task_predictor`, и учится на размеченных концептах. Разметки у CIFAR-10 нет, поэтому поставим вместо нее бинарные метки из того же CLIP: концепт $j$ «есть» на снимке, если его косинус выше 90-го перцентиля этого концепта по обучающим снимкам. Концепты — те же, что в горлышке Label-free CBM. Архитектура — та же, что в той тетради, с размерностями под 512 признаков и 10 классов; обучение совместное (joint): BCE по концептам плюс $0.2\times$ кросс-энтропия по классу, как в первом примере той тетради.

In [ ]:
class Concept_Bottleneck_Model(torch.nn.Module):   # как в PRACTICE_cbm_cem

    def __init__(self, input_size, hidden_size, concept_size, len_out):
        super().__init__()
        self.concept_encoder = torch.nn.Sequential(torch.nn.Linear(input_size, hidden_size), torch.nn.LeakyReLU(),
                                                   torch.nn.Linear(hidden_size, concept_size), torch.nn.Sigmoid())
        self.task_predictor = torch.nn.Sequential(torch.nn.Linear(concept_size, hidden_size), torch.nn.LeakyReLU(),
                                                  torch.nn.Linear(hidden_size, len_out))


T_lf = encode_text(lf_names)
P_train, P_test = X['train'] @ T_lf.T, X['test'] @ T_lf.T
q90 = P_train.quantile(0.9, dim=0)
L_train, L_test = (P_train > q90).float(), (P_test > q90).float()

torch.manual_seed(SEED)
cbm = Concept_Bottleneck_Model(512, 64, len(lf_names), 10).to(device)
opt = torch.optim.AdamW(cbm.parameters(), lr=1e-3)
for epoch in range(500):
    opt.zero_grad()
    c_pred = cbm.concept_encoder(B['train'])
    loss = F.binary_cross_entropy(c_pred, L_train) + 0.2 * F.cross_entropy(cbm.task_predictor(c_pred), y['train'])
    loss.backward()
    opt.step()
with torch.no_grad():
    c_pred = cbm.concept_encoder(B['test'])
    task_acc = (cbm.task_predictor(c_pred).argmax(1) == y['test']).float().mean().item()
    concept_acc = ((c_pred > 0.5) == L_test.bool()).float().mean().item()
results['CBM из PRACTICE_cbm_cem, метки концептов из CLIP'] = (
    task_acc, sum(int((p != 0).sum()) for p in cbm.task_predictor.parameters() if p.dim() == 2))
print(f'точность концептов {concept_acc:.4f} при доле нулей в метках {1 - L_test.mean().item():.4f}')

In [ ]:
table = pd.DataFrame([{'модель': k, 'точность на тесте': round(a, 4), 'ненулевых весов последнего слоя': n}
                      for k, (a, n) in results.items()])
table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
colors = ['#7f8c8d', '#7f8c8d', '#7f8c8d', '#2a6f97', '#2a6f97', '#e67e22', '#8e44ad']
labels_en = ['CLIP zero-shot', 'linear over ResNet-18', 'linear over CLIP', 'Label-free CBM, λ=0',
             f'Label-free CBM, λ={LAM}', 'LaBo, k=5', 'CBM (CLIP concept labels)']
for (k, (a, n)), c, lab in zip(results.items(), colors, labels_en):
    if n > 0:
        ax.scatter(n, a, color=c, s=50)
        ax.annotate(lab, (n, a), textcoords='offset points', xytext=(6, -3), fontsize=8)
ax.axhline(results['CLIP zero-shot'][0], color='#7f8c8d', ls='--', lw=1)
ax.text(25, results['CLIP zero-shot'][0] + 0.004, 'CLIP zero-shot (no trained weights)', fontsize=8, color='#7f8c8d')
ax.set_xscale('log')
ax.set_xlim(20, 20000)
ax.set_xlabel('nonzero weights of the last layer (log scale)')
ax.set_ylabel('test accuracy')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**Задание 6. По таблице и графику: какой из методов с концептами дает последний слой с наименьшим числом ненулевых весов, на сколько пунктов точности он уступает линейному классификатору над признаками той же сети-основы ResNet-18 и почему LaBo можно сравнивать по точности не с ResNet-18, а только с CLIP?**

**Ваш ответ:**

## 7. Где ломается

**Название класса в горлышке.** Уберем фильтр сходства с названием класса (`class_cut=1.01` его выключает) и соберем Label-free CBM заново. Если в списке остаются `a cat`, `aircraft`, `bird`, сильнейший концепт класса может оказаться просто его названием: точность та же, а объяснение «это кошка, потому что на снимке кошка» ничего не объясняет. Посмотрите и на столбец «с фильтром»: `a cat bed`, `a deer stand`, `a dog bowl` фильтр пропустил, потому что косинус с названием класса у них ниже 0.90, но название класса в них все равно стоит. Фильтр по косинусу ловит почти точные копии названия, а не концепты, которые содержат название.

In [ ]:
leak_names, C_leak, _ = concept_layer(presence_filter(text_filters(concepts, class_cut=1.01), verbose=False), verbose=False)
W_leak, b_leak = fit_sparse(C_leak['train'], y['train'], LAM)
print(f'концептов {len(leak_names)}, точность {accuracy(C_leak["test"], W_leak, b_leak, y["test"]):.4f}, ненулевых весов {int((W_leak != 0).sum())}')
pd.DataFrame({'с фильтром': [lf_names[W_lf[i].argmax()] for i in range(10)],
              'без фильтра': [leak_names[W_leak[i].argmax()] for i in range(10)]}, index=CLASSES)

**Что CLIP не видит на снимке 32×32.** Фильтр присутствия в разделе 2 выбросил `headlights`, `fur`, `a steering wheel`, `side windows`: GPT-3 описывала класс по фотографии в хорошем разрешении, а на снимке CIFAR-10 эти детали занимают несколько пикселей, и косинус CLIP с ними не поднимается выше 0.25 ни на одном обучающем снимке. Список концептов от языковой модели не знает, в каком разрешении будут данные.

**Вес уходит с языкового априорного знания.** В разделе 5 у LaBo в начале обучения около 0.94 веса строки лежало на своих концептах, а после обучения в прогоне решения у семи классов из десяти меньше половины (от 0.29 у `automobile` до 0.48 у `deer`, см. вывод раздела 5): классификатор берет любые концепты, которые помогают, и на тепловой карте у каждой строки видны веса на концептах, отобранных для других классов. Объяснение LaBo читается по матрице $A$, а не по исходному отбору.

**Значение концепта равно сходству, которое считает CLIP.** На картинке раздела 4 видно, на каких снимках концепт сильнее всего активен; совпадают ли они с тем, что назвала бы разметчица, проверяют только глазами. Точность концептов у CBM в разделе 6 близка к доле нулей в метках: бинарные метки из CLIP редки, и предсказать «нет» почти везде уже дает высокую точность концептов. Высокая точность концептов без разметки человеком ничего не говорит о том, правильны ли концепты.

## Что унести из тетради

- **Label-free CBM и LaBo заменяют разметку концептов парой готовых моделей.** Языковая модель пишет список, CLIP оценивает, насколько концепт виден на снимке. Label-free CBM учит проекцию признаков сети-основы повторять эти оценки, LaBo берет их прямо как горлышко.
- **Фильтры концептов — часть метода, а не уборка.** Без фильтра сходства с названием класса горлышко пропускает само название, и объяснение вырождается; фильтр по косинусу при этом не ловит концепты, в которых название стоит внутри (`a deer stand`).
- **Разреженный последний слой короче и почти так же точен.** Elastic net с проксимальным шагом при $\lambda=0.003$ оставляет у класса около двух десятков концептов из сотни, а точность на тесте падает примерно на 2 пункта; точность и число ненулевых весов при других $\lambda$ — в таблице раздела 4.
- **У LaBo априорное знание задает только начало обучения.** После обучения вес смещается на чужие концепты, и объяснение надо читать по обученной матрице.
- **Без разметки человеком качество концептов не измерить.** И точность концептов, и картинки с самыми активными снимками говорят о том, какое сходство с концептом считает CLIP.

Источники:
1. [Oikarinen et al., 2023. Label-Free Concept Bottleneck Models](https://arxiv.org/abs/2304.06129), код: [Trustworthy-ML-Lab/Label-free-CBM](https://github.com/Trustworthy-ML-Lab/Label-free-CBM)
2. [Yang et al., 2023. Language in a Bottle: Language Model Guided Concept Bottlenecks for Interpretable Image Classification](https://arxiv.org/abs/2211.11158), код: [YueYANG1996/LaBo](https://github.com/YueYANG1996/LaBo)
3. [Koh et al., 2020. Concept Bottleneck Models](https://arxiv.org/abs/2007.04612)
4. [Radford et al., 2021. Learning Transferable Visual Models From Natural Language Supervision (CLIP)](https://arxiv.org/abs/2103.00020)